# EV purchases (S6E9) — pipeline v3: partition bagging + logistic regression + honest blend

v2 (LB 0.94635) → v3 changes:

1. **Partition bagging** — the whole 10-fold CV is repeated with several seeds (fold split, inner TE folds and model seed all change). Each model's OOF/test = mean over seeds. Measured on your data: 2 LightGBM seeds 0.946214 → 0.946303.
2. **Logistic regression** member (one-hot discrete columns + log-odds of target encodings) — different model family, adds diversity (≈ +0.00003 in blend tests).
3. **Resumable**: every (model, seed) is saved to `preds/` when finished. Existing v2 files (seed 42) and `v2s7_lgb` (seed 7) are reused, not retrained. If the notebook stops, just run it again.
4. **Optional public OOF libraries** (files in `external/`) join the blend — only files that contain honest OOF predictions.
5. **Nested blend check**: blend weights are fitted on 4/5 of the rows and scored on the remaining 1/5 → an honest estimate of the blend.
6. **All 4 boundary rules** (verified on train before use) + **tie-breaking**.

**Runtime (8-core CPU, approx.)** per seed: LightGBM 17 min · XGBoost 38 min · CatBoost 37 min · LR 3 min.
With `SEEDS = [42, 7, 123, 2024]` and nothing new cached except v2: about **4.5 hours**. Run it overnight.

In [17]:
import os, time, glob, warnings
import numpy as np, pandas as pd
from scipy import sparse
from scipy.special import logit
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
warnings.filterwarnings("ignore")

## Config

In [18]:
TRAIN_PATH, TEST_PATH, ORIG_PATH = "train_cleaned.csv", "test.csv", "ev_adoption.csv"
OUT_DIR = "preds"
TARGET = "Will_Buy_EV"
N_FOLDS, INNER_FOLDS = 10, 5
SEEDS  = [42, 7, 123, 2024, 11, 99]           # partition-bagging seeds (fewer = faster)
MODELS = ["lgb", "cat"]  # remove "xgb"/"cat" if short on time
LR = 0.03                             # learning rate for LightGBM / XGBoost (same as v2)

# Optional public OOF libraries (put the files in ./external/). Missing files are skipped.
EXTERNAL = [
    # (name,        oof file,                              test file)
    ("fusion2",  "external/oof_components.csv",      "external/test_components.csv"),
    ("megayak4", "external/oof_four_views.csv",      "external/test_four_views.csv"),
    ("blamerx",  "external/blamerx_oof.csv",         "external/blamerx_submission.csv"),
    ("lasso6", "external/lasso_oof_predictions.csv", "external/lasso_test_fixed.csv")
]
# Optional hedge: a public submission file to mix with our blend (public-LB only, no OOF!)
HEDGE_FILE, HEDGE_WEIGHT = "external/public_best.csv", 0.5  # e.g. "external/public_best.csv"

CAT_COLS = ["Gender", "City_Type", "Current_Car_Type",
            "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]
INC, COM = "Annual_Income_USD", "Daily_Commute_km"
os.makedirs(OUT_DIR, exist_ok=True)

## Load data + target-free features (same as v2)

In [19]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
orig  = pd.read_csv(ORIG_PATH)
for df in (train, test, orig):
    for c in CAT_COLS:
        df[c] = df[c].astype(str).str.strip()

y = (train.pop(TARGET) == "Yes").astype(int).to_numpy()
train_ids, test_ids = train["id"].to_numpy(), test["id"].to_numpy()
full = pd.concat([train.drop(columns="id"), test.drop(columns="id")], ignore_index=True)
RAW_COLS = list(full.columns)
ntr, nte = len(train), len(test)
tr_rows, te_rows = np.arange(ntr), np.arange(ntr, ntr + nte)

X_base = full.copy()
for c in CAT_COLS:
    X_base[c] = pd.factorize(X_base[c])[0].astype(np.int8)

inc = full[INC].round().astype(np.int64).to_numpy()
com = (full[COM] * 10).round().astype(np.int64).to_numpy()

# digits + flags
for i in range(6):
    X_base[f"inc_d{i}"] = ((inc // 10**i) % 10).astype(np.int8)
X_base["com_d_dec"] = (com % 10).astype(np.int8)
X_base["com_d1"]    = ((com // 10) % 10).astype(np.int8)
X_base["inc_mod100"]  = (inc % 100 == 0).astype(np.int8)
X_base["inc_mod1000"] = (inc % 1000 == 0).astype(np.int8)
X_base["inc_is_30k"]  = (inc == 30000).astype(np.int8)
X_base["com_is_int"]  = (com % 10 == 0).astype(np.int8)
X_base["com_is_5"]    = (com == 50).astype(np.int8)
DIGIT_COLS = ["inc_d1", "inc_d2", "inc_d3", "inc_d4", "com_d_dec", "com_d1"]

# value statistics vs. the original dataset
def value_stats(keys, orig_keys, scale, prefix):
    u, inv, cnt = np.unique(keys, return_inverse=True, return_counts=True)
    uo, cnto = np.unique(orig_keys, return_counts=True)
    pos = np.clip(np.searchsorted(uo, u), 0, len(uo) - 1)
    found = uo[pos] == u
    cnt_orig = np.where(found, cnto[pos], 0)
    share_all, share_orig = cnt / len(keys), cnt_orig / len(orig_keys)
    lift = share_all / np.where(share_orig > 0, share_orig, 1 / len(orig_keys))
    p = np.searchsorted(uo, u)
    lo = uo[np.clip(p - 1, 0, len(uo) - 1)]
    hi = uo[np.clip(p, 0, len(uo) - 1)]
    d_lo, d_hi = (u - lo) / scale, (hi - u) / scale
    dist = np.where(np.abs(d_lo) <= np.abs(d_hi), -d_lo, d_hi)
    dist[found] = 0.0
    rank = cnt.argsort().argsort() / len(cnt)
    return pd.DataFrame({
        f"{prefix}_cnt": cnt[inv].astype(np.int32),
        f"{prefix}_cnt_orig": cnt_orig[inv].astype(np.int16),
        f"{prefix}_lift": np.log1p(lift[inv]).astype(np.float32),
        f"{prefix}_novel": (~found)[inv].astype(np.int8),
        f"{prefix}_dist_orig": dist[inv].astype(np.float32),
        f"{prefix}_freqrank": rank[inv].astype(np.float32),
    })

o_inc = orig[INC].dropna().round().astype(np.int64).to_numpy()
o_com = (orig[COM].dropna() * 10).round().astype(np.int64).to_numpy()
X_base = pd.concat([X_base, value_stats(inc, o_inc, 1, "inc"), value_stats(com, o_com, 10, "com")], axis=1)
print(f"train {ntr} | test {nte} | base features {X_base.shape[1]}")

train 668665 | test 286571 | base features 38


## Target-encoding specs + nested TE (same as v2, but the seed is a parameter)

In [20]:
te_specs = {}
def add_spec(name, key, m):
    codes, uniq = pd.factorize(key)
    te_specs[name] = (codes.astype(np.int64), len(uniq), m)

add_spec("te_inc", inc, 3)
for w in [100, 500, 2000]:
    add_spec(f"te_inc_b{w}", inc // w, 3)
add_spec("te_com", com, 3)
for w in [10, 50]:
    add_spec(f"te_com_b{w}", com // w, 3)
OTHER = [c for c in RAW_COLS if c not in (INC, COM)]
for m in (10, 100):
    for c in OTHER + DIGIT_COLS:
        add_spec(f"te_{c}_m{m}", X_base[c].to_numpy(), m)
    add_spec(f"te_inc_b1000_m{m}", inc // 1000, m)

def te_map(codes_fit, y_fit, codes_apply, n, m):
    prior = y_fit.mean()
    s = np.bincount(codes_fit, weights=y_fit, minlength=n)
    c = np.bincount(codes_fit, minlength=n)
    return ((s + prior * m) / (c + m))[codes_apply].astype(np.float32)

def build_te(fit_idx, apply_idx_list, seed):
    y_fit = y[fit_idx]
    inner = list(KFold(INNER_FOLDS, shuffle=True, random_state=seed).split(fit_idx))
    fit_out, apply_out = {}, [{} for _ in apply_idx_list]
    for name, (codes, n, m) in te_specs.items():
        c_fit = codes[fit_idx]
        col = np.empty(len(fit_idx), dtype=np.float32)
        for a, b in inner:
            col[b] = te_map(c_fit[a], y_fit[a], c_fit[b], n, m)
        fit_out[name] = col
        for k, idx in enumerate(apply_idx_list):
            apply_out[k][name] = te_map(c_fit, y_fit, codes[idx], n, m)
    return pd.DataFrame(fit_out), [pd.DataFrame(d) for d in apply_out]

def make_xy(fit_idx, apply_idx_list, seed):
    te_fit, te_apply = build_te(fit_idx, apply_idx_list, seed)
    X_fit = pd.concat([X_base.iloc[fit_idx].reset_index(drop=True), te_fit], axis=1)
    X_apply = [pd.concat([X_base.iloc[i].reset_index(drop=True), t], axis=1)
               for i, t in zip(apply_idx_list, te_apply)]
    return X_fit, X_apply
print("target encodings:", len(te_specs))

target encodings: 43


## Models
LightGBM / XGBoost / CatBoost are **identical to v2** (so seed 42 = your v2 files).
New: **logistic regression** — a linear (additive) model, matching how the data was generated:
discrete columns one-hot encoded, target encodings as log-odds, a few scaled numerics.

In [21]:
def fit_lgb(Xtr, ytr, Xva, yva, seed):
    m = lgb.LGBMClassifier(
        objective="binary", n_estimators=20000, learning_rate=LR,
        max_depth=5, num_leaves=32, min_child_samples=10,
        colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
        reg_alpha=0.071, reg_lambda=2.0, max_bin=1024,
        random_state=seed, n_jobs=-1, verbose=-1)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc",
          callbacks=[lgb.early_stopping(300, verbose=False)])
    return m, m.best_iteration_

def fit_xgb(Xtr, ytr, Xva, yva, seed):
    m = xgb.XGBClassifier(
        n_estimators=20000, learning_rate=LR, tree_method="hist",
        max_depth=5, min_child_weight=10, colsample_bytree=0.3, subsample=0.8,
        reg_alpha=0.071, reg_lambda=2.0, max_bin=1024,
        eval_metric="auc", early_stopping_rounds=300,
        random_state=seed, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m, m.best_iteration

def fit_cat(Xtr, ytr, Xva, yva, seed):
    m = CatBoostClassifier(
        loss_function="Logloss", eval_metric="AUC", iterations=20000,
        learning_rate=0.08, depth=5, rsm=0.3, l2_leaf_reg=2.0, border_count=1024,
        early_stopping_rounds=300, random_seed=seed, verbose=0, thread_count=-1)
    m.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True)
    return m, m.get_best_iteration()

LR_ONEHOT = CAT_COLS + ["Environmental_Concern_Level", "Age", "Number_of_Cars_Owned",
                        "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
                        "inc_d0", "inc_d1", "inc_d2", "inc_d3", "com_d_dec", "com_d1",
                        "inc_is_30k", "com_is_5", "inc_novel", "inc_mod100"]
LR_NUM = ["Annual_Income_USD", "Daily_Commute_km", "inc_lift", "com_lift"]

class LRModel:
    """Logistic regression on one-hot discrete columns + log-odds of the target encodings."""
    def _dense(self, X):
        te_cols = [c for c in X.columns if c.startswith("te_")]
        T = logit(np.clip(X[te_cols].to_numpy(np.float64), 1e-5, 1 - 1e-5))
        N = X[LR_NUM].to_numpy(np.float64)
        C = np.log1p(X[["inc_cnt", "com_cnt"]].to_numpy(np.float64))
        return np.column_stack([T, N, C])
    def _matrix(self, X):
        return sparse.hstack([self.enc.transform(X[LR_ONEHOT]),
                              sparse.csr_matrix(self.sc.transform(self._dense(X)))]).tocsr()
    def fit(self, X, yv):
        self.enc = OneHotEncoder(handle_unknown="ignore", min_frequency=20).fit(X[LR_ONEHOT])
        self.sc = StandardScaler().fit(self._dense(X))
        self.m = LogisticRegression(C=0.5, max_iter=400).fit(self._matrix(X), yv)
        return self
    def predict_proba(self, X):
        return self.m.predict_proba(self._matrix(X))

def fit_lr(Xtr, ytr, Xva, yva, seed):
    m = LRModel().fit(Xtr, ytr)
    return m, m.m.n_iter_[0]

FITTERS = {"lgb": fit_lgb, "xgb": fit_xgb, "cat": fit_cat, "lr": fit_lr}

## Partition-bagging loop (resumable)
For every seed: new 10-fold split, new inner TE folds, new model seed. Each finished (model, seed)
is saved as `preds/oof_v3_<model>_s<seed>.npy` / `test_...`. Already-finished pairs are skipped;
seed 42 LightGBM/XGBoost/CatBoost reuse your v2 files and seed 7 LightGBM reuses `v2s7_lgb`.

In [22]:
LEGACY = {("lgb", 42): "v2_lgb", ("xgb", 42): "v2_xgb", ("cat", 42): "v2_cat", ("lgb", 7): "v2s7_lgb"}
def tag(model, seed):
    return LEGACY.get((model, seed), f"v3_{model}_s{seed}")
def done(model, seed):
    t = tag(model, seed)
    return os.path.exists(f"{OUT_DIR}/oof_{t}.npy") and os.path.exists(f"{OUT_DIR}/test_{t}.npy")

T0 = time.time()
for seed in SEEDS:
    todo = [m for m in MODELS if not done(m, seed)]
    if not todo:
        print(f"seed {seed}: all models cached")
        continue
    print(f"\n===== seed {seed}: training {todo} =====")
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed)
    oof = {k: np.zeros(ntr) for k in todo}
    tst = {k: np.zeros(nte) for k in todo}
    for f, (tr, va) in enumerate(skf.split(tr_rows, y), 1):
        t0 = time.time()
        Xtr, (Xva, Xte) = make_xy(tr, [va, te_rows], seed)
        line = f"seed {seed} fold {f:2d}/{N_FOLDS} (features {time.time()-t0:.0f}s)"
        for k in todo:
            t1 = time.time()
            model, it = FITTERS[k](Xtr, y[tr], Xva, y[va], seed)
            oof[k][va] = model.predict_proba(Xva)[:, 1]
            tst[k] += model.predict_proba(Xte)[:, 1] / N_FOLDS
            line += f" | {k} {roc_auc_score(y[va], oof[k][va]):.5f} ({time.time()-t1:.0f}s)"
        print(line, flush=True)
    for k in todo:
        np.save(f"{OUT_DIR}/oof_{tag(k, seed)}.npy", oof[k]); np.save(f"{OUT_DIR}/test_{tag(k, seed)}.npy", tst[k])
        print(f"saved {tag(k, seed)}: OOF {roc_auc_score(y, oof[k]):.6f}")
print(f"\ntraining done in {(time.time()-T0)/60:.0f} min")

seed 42: all models cached
seed 7: all models cached
seed 123: all models cached
seed 2024: all models cached

===== seed 11: training ['lgb', 'cat'] =====
seed 11 fold  1/10 (features 11s) | lgb 0.94640 (145s) | cat 0.94634 (240s)
seed 11 fold  2/10 (features 9s) | lgb 0.94586 (120s) | cat 0.94594 (269s)
seed 11 fold  3/10 (features 9s) | lgb 0.94600 (122s) | cat 0.94590 (140s)
seed 11 fold  4/10 (features 7s) | lgb 0.94643 (94s) | cat 0.94636 (307s)
seed 11 fold  5/10 (features 11s) | lgb 0.94470 (163s) | cat 0.94458 (254s)
seed 11 fold  6/10 (features 9s) | lgb 0.94606 (84s) | cat 0.94605 (209s)
seed 11 fold  7/10 (features 8s) | lgb 0.94685 (112s) | cat 0.94687 (245s)
seed 11 fold  8/10 (features 13s) | lgb 0.94764 (117s) | cat 0.94760 (231s)
seed 11 fold  9/10 (features 8s) | lgb 0.94498 (104s) | cat 0.94506 (272s)
seed 11 fold 10/10 (features 8s) | lgb 0.94746 (122s) | cat 0.94746 (231s)
saved v3_lgb_s11: OOF 0.946224
saved v3_cat_s11: OOF 0.946203

===== seed 99: training ['lgb'

## Bags: average each model over its seeds
Averaging predictions from different fold splits reduces the variance coming from the random
fold split, the inner target-encoding folds and the model seed.

In [23]:
def to_rank(p):
    return rankdata(p) / len(p)

bag_oof, bag_tst, rows = {}, {}, []
for k in MODELS:
    seeds = [s for s in SEEDS if done(k, s)]
    if not seeds:
        continue
    O = [np.load(f"{OUT_DIR}/oof_{tag(k, s)}.npy") for s in seeds]
    T = [np.load(f"{OUT_DIR}/test_{tag(k, s)}.npy") for s in seeds]
    bag_oof[f"bag_{k}"], bag_tst[f"bag_{k}"] = np.mean(O, 0), np.mean(T, 0)
    rows.append(dict(model=k, seeds=len(seeds),
                     single_mean=np.mean([roc_auc_score(y, o) for o in O]),
                     bag=roc_auc_score(y, bag_oof[f"bag_{k}"])))
board = pd.DataFrame(rows); board["gain"] = board.bag - board.single_mean
print(board.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

R = pd.DataFrame({k: to_rank(v) for k, v in bag_oof.items()})
print("\nSpearman correlation between bags:")
print(R.corr().round(4).to_string())

model  seeds  single_mean      bag     gain
  lgb      6     0.946221 0.946360 0.000139
  cat      6     0.946211 0.946373 0.000162

Spearman correlation between bags:
         bag_lgb  bag_cat
bag_lgb   1.0000   0.9984
bag_cat   0.9984   1.0000


## Candidate pool: our bags + v1 models + optional public OOF libraries
A public file is only used if it has an **OOF file** (predictions for the training rows).
Columns are matched by `id`; label columns (`Will_Buy_EV` in an OOF file, `fold`, `id`) are skipped.

In [24]:
cand_oof, cand_tst = dict(bag_oof), dict(bag_tst)
for k in ["cat", "lgb", "mlp"]:                      # v1 models (different features -> some diversity)
    if os.path.exists(f"{OUT_DIR}/oof_{k}.npy") and os.path.exists(f"{OUT_DIR}/test_{k}.npy"):
        cand_oof[f"v1_{k}"], cand_tst[f"v1_{k}"] = np.load(f"{OUT_DIR}/oof_{k}.npy"), np.load(f"{OUT_DIR}/test_{k}.npy")
OWN = list(cand_oof)

META = {"id", "fold", "target", "y", "label", "unnamed: 0"}
def load_external(name, oof_path, test_path):
    if not (os.path.exists(oof_path) and os.path.exists(test_path)):
        print(f"[skip] {name}: files not found"); return
    o = pd.read_csv(oof_path).set_index("id").reindex(train_ids)
    t = pd.read_csv(test_path).set_index("id").reindex(test_ids)
    o_cols = [c for c in o.columns if c.lower() not in META and c != TARGET]
    t_cols = [c for c in t.columns if c.lower() not in META]
    pairs = [(c, c) for c in o_cols if c in t_cols]
    if not pairs and len(o_cols) == 1 and len(t_cols) == 1:   # e.g. oof.csv 'pred' <-> submission.csv 'Will_Buy_EV'
        pairs = [(o_cols[0], t_cols[0])]
    for oc, tc in pairs:
        ov, tv = o[oc].to_numpy(float), t[tc].to_numpy(float)
        if not (np.isfinite(ov).all() and np.isfinite(tv).all()):
            print(f"[skip] {name}:{oc} has missing rows after id alignment"); continue
        a = roc_auc_score(y, ov)
        if not (0.90 < a < 0.99):
            print(f"[skip] {name}:{oc} OOF AUC {a:.4f} looks wrong (label column or broken file)"); continue
        cand_oof[f"{name}:{oc}"], cand_tst[f"{name}:{oc}"] = ov, tv
        print(f"[ok]   {name}:{oc:28s} OOF {a:.6f}")

for name, op, tp in EXTERNAL:
    load_external(name, op, tp)
EXT = [k for k in cand_oof if k not in OWN]
print(f"\ncandidates: {len(OWN)} own + {len(EXT)} external")

[ok]   fusion2:C1_chinzo_lgbm               OOF 0.946342
[ok]   fusion2:A1_najiama_xgb               OOF 0.946321
[ok]   fusion2:D_megayak_liftwin            OOF 0.946309
[ok]   fusion2:E_bulsah_mrte                OOF 0.946291
[ok]   fusion2:A2_najiama_lgbm              OOF 0.946291
[ok]   fusion2:C2_chinzo_catboost           OOF 0.946276
[ok]   fusion2:C3_chinzo_linear             OOF 0.945678
[ok]   fusion2:fusion                       OOF 0.946398
[ok]   megayak4:A_lgbm_triple_te_3seed       OOF 0.946281
[ok]   megayak4:B_xgb_same_features          OOF 0.946258
[ok]   megayak4:C_lgbm_windows_lift_nodigit  OOF 0.946223
[ok]   megayak4:D_lgbm_alt_ladder_noexact    OOF 0.946077
[ok]   megayak4:ensemble_ABCD                OOF 0.946339
[ok]   blamerx:pred                         OOF 0.946208
[ok]   lasso6:lasso_oof                    OOF 0.946349

candidates: 5 own + 15 external


## Hill-climbing blend + nested check
`hill_climb` starts from the best single candidate and greedily adds (with replacement) whatever
raises OOF AUC. The **nested check** fits weights on 4/5 of the rows and scores the other 1/5,
five times — if the blend wins there, the gain is real and not just fitted to the OOF.

In [25]:
def fast_auc(yv, s):
    r = rankdata(s); npos = yv.sum()
    return (r[yv == 1].sum() - npos * (npos + 1) / 2) / (npos * (len(yv) - npos))

def hill_climb(P, yv, max_steps=60):
    k = P.shape[1]
    single = np.array([fast_auc(yv, P[:, j]) for j in range(k)])
    counts = np.zeros(k); j0 = int(single.argmax()); counts[j0] = 1
    cur, cur_auc = P[:, j0].copy(), single[j0]
    for _ in range(max_steps):
        n = counts.sum()
        trial = [fast_auc(yv, (cur * n + P[:, j]) / (n + 1)) for j in range(k)]
        j = int(np.argmax(trial))
        if trial[j] <= cur_auc + 1e-7:
            break
        counts[j] += 1; cur, cur_auc = (cur * n + P[:, j]) / (n + 1), trial[j]
    return counts / counts.sum(), single

block = np.zeros(ntr, dtype=int)                      # 5 stratified blocks for the nested check
for b, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(tr_rows, y)):
    block[va] = b

def blend(names, label):
    P = np.column_stack([to_rank(cand_oof[n]) for n in names]).astype(np.float32)
    gains = []
    for b in range(5):
        fit, hold = block != b, block == b
        w, single = hill_climb(P[fit], y[fit])
        gains.append(fast_auc(y[hold], P[hold] @ w) - fast_auc(y[hold], P[hold, int(single.argmax())]))
    w, single = hill_climb(P, y)
    W = {n: float(v) for n, v in zip(names, w) if v > 0}
    oof_b = sum(v * to_rank(cand_oof[n]) for n, v in W.items())
    tst_b = sum(v * to_rank(cand_tst[n]) for n, v in W.items())
    print(f"--- {label} ---")
    print("weights:", {n: round(v, 3) for n, v in W.items()})
    print(f"best single {single.max():.6f} ({names[int(single.argmax())]}) -> blend OOF {fast_auc(y, oof_b):.6f}")
    print(f"nested check: blend beats best single in {sum(g > 0 for g in gains)}/5 blocks, "
          f"mean held-out gain {np.mean(gains):+.6f}\n")
    return oof_b, tst_b

oof_own, tst_own = blend(OWN, "own models only")
oof_all, tst_all = blend(OWN + EXT, "own + public OOF") if EXT else (oof_own, tst_own)

--- own models only ---
weights: {'bag_lgb': 0.5, 'bag_cat': 0.5}
best single 0.946373 (bag_cat) -> blend OOF 0.946397
nested check: blend beats best single in 5/5 blocks, mean held-out gain +0.000022

--- own + public OOF ---
weights: {'bag_cat': 0.286, 'fusion2:C1_chinzo_lgbm': 0.143, 'fusion2:fusion': 0.143, 'megayak4:ensemble_ABCD': 0.143, 'blamerx:pred': 0.143, 'lasso6:lasso_oof': 0.143}
best single 0.946398 (fusion2:fusion) -> blend OOF 0.946459
nested check: blend beats best single in 5/5 blocks, mean held-out gain +0.000059



## Boundary rules + tie-breaking → submissions
Each rule is checked on train first and only used if it is 100% pure (all Yes or all No).
Then ties are broken by the bagged LightGBM prediction.

In [26]:
def rule_masks(df):
    inc_ = df[INC].to_numpy(); km = df[COM].to_numpy()
    env1 = df["Environmental_Concern_Level"].to_numpy() == 1
    nosub = df["Subsidy_Available"].to_numpy() == "No"
    anx_mh = df["Range_Anxiety_Level"].isin(["Medium", "High"]).to_numpy()
    return {"income >= 170,537": (inc_ >= 170537, +10.0),
            "31,004 <= income <= 41,970": ((inc_ >= 31004) & (inc_ <= 41970), -10.0),
            "commute >= 83 km": (km >= 83, -5.0),
            "30k & no subsidy & (env1 | anxiety M/H)": ((inc_ == 30000) & nosub & (env1 | anx_mh), -5.0)}

tr_df, te_df = full.iloc[:ntr].reset_index(drop=True), full.iloc[ntr:].reset_index(drop=True)
m_tr, m_te = rule_masks(tr_df), rule_masks(te_df)
USE_RULES = []
for name, (mask, s) in m_tr.items():
    rate = y[mask].mean() if mask.sum() else np.nan
    pure = (s > 0 and rate == 1.0) or (s < 0 and rate == 0.0)
    print(f"{name:42s} train rows {mask.sum():6d} buy rate {rate:.4f} test rows {m_te[name][0].sum():5d} -> {'USE' if pure else 'skip'}")
    if pure: USE_RULES.append(name)

def finalize(score, masks, tiebreak):
    s = score + sum(np.where(masks[n][0], masks[n][1], 0.0) for n in USE_RULES)
    order = np.lexsort((tiebreak, s))
    r = np.empty(len(s)); r[order] = np.arange(1, len(s) + 1)
    return (r - 0.5) / len(s)

tb = "bag_lgb" if "bag_lgb" in bag_oof else list(bag_oof)[0]   # tie-breaker model
tb_oof, tb_tst = to_rank(bag_oof[tb]), to_rank(bag_tst[tb])
final = {}
for label, o, t in [("own", oof_own, tst_own), ("blend", oof_all, tst_all)]:
    fo, ft = finalize(o, m_tr, tb_oof), finalize(t, m_te, tb_tst)
    print(f"{label:5s}: OOF {fast_auc(y, o):.6f} -> with rules + tie-break {fast_auc(y, fo):.6f}")
    final[label] = ft
    pd.DataFrame({"id": test_ids, TARGET: ft}).to_csv(f"submission_v4_{label}.csv", index=False)
print("saved submission_v4_own.csv and submission_v4_blend.csv")

income >= 170,537                          train rows    393 buy rate 1.0000 test rows   156 -> USE
31,004 <= income <= 41,970                 train rows   1257 buy rate 0.0000 test rows   494 -> USE
commute >= 83 km                           train rows    186 buy rate 0.0000 test rows    81 -> USE
30k & no subsidy & (env1 | anxiety M/H)    train rows   7157 buy rate 0.0000 test rows  3002 -> USE
own  : OOF 0.946397 -> with rules + tie-break 0.946402
blend: OOF 0.946459 -> with rules + tie-break 0.946465
saved submission_v4_own.csv and submission_v4_blend.csv


## Optional hedge (public-leaderboard only)
Mixes a public **submission** file (no OOF, so it cannot be validated) with our blend.
Only use it as your *second* final pick. Set `HEDGE_FILE` in the config to enable.

In [27]:
if HEDGE_FILE and os.path.exists(HEDGE_FILE):
    h = pd.read_csv(HEDGE_FILE).set_index("id").reindex(test_ids)[TARGET].to_numpy(float)
    assert np.isfinite(h).all(), "hedge file ids do not match test ids"
    print(f"spearman(hedge file, our blend) = {np.corrcoef(to_rank(h), final['blend'])[0, 1]:.5f}")
    mix = HEDGE_WEIGHT * to_rank(h) + (1 - HEDGE_WEIGHT) * final["blend"]
    hedge = finalize(mix, m_te, tb_tst)
    pd.DataFrame({"id": test_ids, TARGET: hedge}).to_csv("submission_v4_hedge.csv", index=False)
    print("saved submission_v4_hedge.csv")
else:
    print("no hedge file set - skipped")

spearman(hedge file, our blend) = 0.99909
saved submission_v4_hedge.csv
